# Step 1: Data Audit and Dataset Characterization

This notebook characterizes the primary PPG dataset through the Step 1 core pipeline. It does not parse CSV files directly, preprocess signals, change labels, or recreate segmentation and eligibility logic.

Frozen labels: `0 = Awake`, `1 = Drowsy`.

## 1. Setup

Paths, publication style, and state encoding are defined once.

In [ ]:
import logging
import sys
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    """Find the phase1 project root."""
    for candidate in (start, *start.parents):
        if (candidate / "src" / "dataloader").is_dir():
            return candidate
    raise FileNotFoundError("Cannot locate the phase1 project root")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SOURCE_DIR = PROJECT_ROOT / "src"
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

from dataloader import audit_dataset  # noqa: E402
from dataloader.schema import EXPECTED_FS, WINDOW_DURATIONS  # noqa: E402

DATASET_DIR = PROJECT_ROOT / "dataset" / "dhdata"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "data_audit"
FIGURE_DIR = OUTPUT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

AWAKE_COLOR = "#0072B2"
DROWSY_COLOR = "#D55E00"
PAIRED_COLOR = "#009E73"
NEUTRAL_COLOR = "#4D4D4D"
STATE_COLORS = {
    "Awake": AWAKE_COLOR,
    "Drowsy": DROWSY_COLOR,
    "Paired": PAIRED_COLOR,
}
SINGLE_COLUMN = (3.5, 2.7)
DOUBLE_COLUMN = (7.2, 4.2)
mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 8,
    "axes.labelsize": 9,
    "axes.linewidth": 0.8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 8,
    "lines.linewidth": 1.2,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})


def save_figure(figure: mpl.figure.Figure, name: str) -> None:
    """Export one vector and one raster figure."""
    figure.savefig(FIGURE_DIR / f"{name}.pdf")
    figure.savefig(FIGURE_DIR / f"{name}.png", dpi=300)


print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset: {DATASET_DIR}")
print(f"Audit output: {OUTPUT_DIR}")

## 2. Run the Core Audit

All downstream tables and plots use the returned core outputs.

In [ ]:
audit_result = audit_dataset(DATASET_DIR, OUTPUT_DIR)
inventory = audit_result.inventory.copy()
summary = audit_result.summary.copy()
session_reports = list(audit_result.session_reports)
print(f"Audited sessions: {len(session_reports)}")
print(f"Inventory: {audit_result.output_paths['inventory']}")
print(f"Summary: {audit_result.output_paths['summary']}")

## 3. Dataset Overview

In [ ]:
overview_table = pd.DataFrame({
    "Metric": [
        "Sessions",
        "Samples",
        "Recording duration (h)",
        "Session duration min / median / max (min)",
        "Sampling frequency min / median / max (Hz)",
        "Sessions with Awake / Drowsy / both",
    ],
    "Value": [
        f"{summary['total_sessions']:,}",
        f"{summary['total_samples']:,}",
        f"{summary['total_recording_hours']:.2f}",
        (
            f"{summary['session_duration_min'] / 60:.2f} / "
            f"{summary['session_duration_median'] / 60:.2f} / "
            f"{summary['session_duration_max'] / 60:.2f}"
        ),
        (
            f"{summary['sampling_rate_min']:.2f} / "
            f"{summary['sampling_rate_median']:.2f} / "
            f"{summary['sampling_rate_max']:.2f}"
        ),
        (
            f"{summary['sessions_with_awake']} / "
            f"{summary['sessions_with_drowsy']} / "
            f"{summary['sessions_with_both_states']}"
        ),
    ],
})
display(overview_table.style.hide(axis="index"))

## 4. Data Quality Report

Counts aggregate explicit core fields and issue messages. No session is removed.

In [ ]:
def count_nulls(report: dict) -> int:
    """Count null values in one core report."""
    return sum(
        item["null_count"]
        for item in report["missing_values"].values()
    )


def count_infinity(report: dict, sign: str) -> int:
    """Count signed infinite values."""
    return sum(
        item[sign]
        for item in report["infinity_counts"].values()
    )


def has_sampling_issue(report: dict) -> bool:
    """Read sampling inconsistency from core issues."""
    return any(
        "sampling rate" in issue.lower()
        for issue in report["issues"]
    )


quality_summary = pd.DataFrame({
    "Measure": [
        "PASS / WARNING / FAIL sessions",
        "Null / NaN values",
        "Positive / negative infinity values",
        "Duplicate rows",
        "Duplicate timestamps",
        "Non-monotonic timestamps",
        "Time gaps",
        "Invalid labels",
        "Empty columns",
        "Sampling-rate inconsistencies",
    ],
    "Count": [
        (
            f"{summary['sessions_pass']} / "
            f"{summary['sessions_warning']} / "
            f"{summary['sessions_fail']}"
        ),
        sum(count_nulls(report) for report in session_reports),
        (
            f"{sum(count_infinity(report, 'positive_inf') for report in session_reports)} / "
            f"{sum(count_infinity(report, 'negative_inf') for report in session_reports)}"
        ),
        sum(report["duplicate_rows"] for report in session_reports),
        sum(report["n_duplicate_timestamps"] for report in session_reports),
        sum(
            report["n_non_monotonic_timestamps"]
            for report in session_reports
        ),
        sum(report["n_time_gaps"] for report in session_reports),
        sum(len(report["invalid_labels"]) for report in session_reports),
        sum(len(report["empty_columns"]) for report in session_reports),
        sum(has_sampling_issue(report) for report in session_reports),
    ],
})
quality_table = pd.DataFrame([
    {
        "Session": report["file_name"],
        "Duration (min)": report["duration_s"] / 60,
        "fs (Hz)": report["estimated_fs"],
        "Null": count_nulls(report),
        "Gaps": report["n_time_gaps"],
        "Invalid labels": len(report["invalid_labels"]),
        "Status": report["status"],
    }
    for report in session_reports
])
display(quality_summary.style.hide(axis="index"))
display(quality_table.style.format({
    "Duration (min)": "{:.2f}",
    "fs (Hz)": "{:.2f}",
}).hide(axis="index"))

## 5. Awake-Drowsy Composition

Every audited session is shown in discovery order.

In [ ]:
state_table = pd.DataFrame([
    {
        "Session": report["file_name"],
        "Awake duration (min)": report["awake_duration_s"] / 60,
        "Drowsy duration (min)": report["drowsy_duration_s"] / 60,
        "Awake (%)": report["awake_percentage"],
        "Drowsy (%)": report["drowsy_percentage"],
    }
    for report in session_reports
])
state_totals = pd.DataFrame({
    "State": ["Awake", "Drowsy"],
    "Duration (h)": [
        summary["total_awake_duration_s"] / 3600,
        summary["total_drowsy_duration_s"] / 3600,
    ],
    "Percentage": [
        summary["awake_percentage"],
        summary["drowsy_percentage"],
    ],
})
display(state_totals.style.format({
    "Duration (h)": "{:.2f}",
    "Percentage": "{:.2f}",
}).hide(axis="index"))
display(state_table.style.format({
    "Awake duration (min)": "{:.2f}",
    "Drowsy duration (min)": "{:.2f}",
    "Awake (%)": "{:.2f}",
    "Drowsy (%)": "{:.2f}",
}).hide(axis="index"))

figure_height = max(4.2, 0.24 * len(state_table) + 1.2)
fig, ax = plt.subplots(figsize=(DOUBLE_COLUMN[0], figure_height))
positions = np.arange(len(state_table))
awake = state_table["Awake duration (min)"]
drowsy = state_table["Drowsy duration (min)"]
ax.barh(positions, awake, color=AWAKE_COLOR, label="Awake")
ax.barh(positions, drowsy, left=awake, color=DROWSY_COLOR, label="Drowsy")
ax.set_yticks(positions, state_table["Session"])
ax.invert_yaxis()
ax.set_xlabel("Labeled recording duration (min)")
ax.set_ylabel("Session")
ax.legend(frameon=False, ncol=2, loc="lower right")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", color="0.9", linewidth=0.6)
ax.set_axisbelow(True)
fig.tight_layout()
save_figure(fig, "state_composition_by_session")
plt.show()

## 6. Continuous State Segments

Only contiguous core segments are used. Segment points are descriptive, not independent population samples.

In [ ]:
segment_table = pd.DataFrame([
    {
        "Session": report["file_name"],
        "Segment": segment["segment_id"],
        "State": segment["state"],
        "Duration (s)": segment["duration_s"],
    }
    for report in session_reports
    for segment in report["segments"]
    if segment["state"] in ("Awake", "Drowsy")
])
segment_summary = pd.DataFrame({
    "Measure": [
        "Total state transitions",
        "Awake segments",
        "Drowsy segments",
        "Longest Awake segment (s)",
        "Longest Drowsy segment (s)",
        "Median Awake segment (s)",
        "Median Drowsy segment (s)",
    ],
    "Value": [
        summary["total_state_transitions"],
        (segment_table["State"] == "Awake").sum(),
        (segment_table["State"] == "Drowsy").sum(),
        segment_table.loc[segment_table["State"] == "Awake", "Duration (s)"].max(),
        segment_table.loc[segment_table["State"] == "Drowsy", "Duration (s)"].max(),
        segment_table.loc[segment_table["State"] == "Awake", "Duration (s)"].median(),
        segment_table.loc[segment_table["State"] == "Drowsy", "Duration (s)"].median(),
    ],
})
transition_table = inventory[["file_name", "n_transitions"]].rename(
    columns={"file_name": "Session", "n_transitions": "Transitions"}
)
display(segment_summary.style.format({"Value": "{:.2f}"}).hide(axis="index"))
display(transition_table.style.hide(axis="index"))

states = ["Awake", "Drowsy"]
segment_values = [
    segment_table.loc[segment_table["State"] == state, "Duration (s)"]
    .dropna().to_numpy()
    for state in states
]
fig, ax = plt.subplots(figsize=SINGLE_COLUMN)
boxplot = ax.boxplot(
    segment_values,
    positions=[1, 2],
    widths=0.45,
    patch_artist=True,
    showfliers=False,
    medianprops={"color": "black", "linewidth": 1.2},
)
for patch, state in zip(boxplot["boxes"], states):
    patch.set_facecolor(STATE_COLORS[state])
    patch.set_alpha(0.35)
for position, state, values in zip([1, 2], states, segment_values):
    offsets = np.linspace(-0.12, 0.12, len(values))
    ax.scatter(
        position + offsets,
        values,
        color=STATE_COLORS[state],
        edgecolor="white",
        linewidth=0.25,
        s=13,
        alpha=0.75,
        zorder=3,
    )
ax.set_xticks([1, 2], states)
ax.set_ylabel("Continuous segment duration (s)")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="0.9", linewidth=0.6)
ax.set_axisbelow(True)
fig.tight_layout()
save_figure(fig, "continuous_segment_duration")
plt.show()

## 7. Window Availability

Eligibility is read directly from the core contiguous-segment checks.

In [ ]:
availability_table = pd.DataFrame([
    {
        "Duration (s)": duration,
        "Awake eligible": int(inventory[f"awake_ge_{duration}"].sum()),
        "Drowsy eligible": int(inventory[f"drowsy_ge_{duration}"].sum()),
        "Paired eligible": int(inventory[f"paired_{duration}"].sum()),
    }
    for duration in WINDOW_DURATIONS
])
display(availability_table.style.hide(axis="index"))

fig, ax = plt.subplots(figsize=SINGLE_COLUMN)
positions = np.arange(len(availability_table))
width = 0.24
for offset, state, column in [
    (-width, "Awake", "Awake eligible"),
    (0, "Drowsy", "Drowsy eligible"),
    (width, "Paired", "Paired eligible"),
]:
    ax.bar(
        positions + offset,
        availability_table[column],
        width=width,
        color=STATE_COLORS[state],
        label=state,
    )
ax.set_xticks(positions, availability_table["Duration (s)"])
ax.set_xlabel("Window duration (s)")
ax.set_ylabel("Eligible sessions (n)")
ax.set_ylim(0, len(inventory) * 1.08)
ax.legend(frameon=False, ncol=3, loc="upper center")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="0.9", linewidth=0.6)
ax.set_axisbelow(True)
fig.tight_layout()
save_figure(fig, "session_availability_by_window_duration")
plt.show()

## 8. Acquisition Consistency

Session observations are retained; 50 Hz is the frozen reference.

In [ ]:
acquisition_table = inventory[[
    "file_name",
    "n_rows",
    "duration_s",
    "estimated_fs",
    "dt_median",
    "dt_std",
]].copy()
acquisition_table["duration_s"] /= 60
acquisition_table.columns = [
    "Session",
    "Samples",
    "Duration (min)",
    "Estimated fs (Hz)",
    "Median dt (s)",
    "dt SD (s)",
]
display(acquisition_table.style.format({
    "Duration (min)": "{:.2f}",
    "Estimated fs (Hz)": "{:.3f}",
    "Median dt (s)": "{:.6f}",
    "dt SD (s)": "{:.6f}",
}).hide(axis="index"))

positions = np.arange(len(acquisition_table))
fig, axes = plt.subplots(2, 1, figsize=(DOUBLE_COLUMN[0], 5.4), sharex=True)
axes[0].plot(
    positions,
    acquisition_table["Duration (min)"],
    color=NEUTRAL_COLOR,
    marker="o",
    markersize=3.5,
)
axes[0].set_ylabel("Recording duration (min)")
axes[1].plot(
    positions,
    acquisition_table["Estimated fs (Hz)"],
    color=NEUTRAL_COLOR,
    marker="o",
    markersize=3.5,
)
axes[1].axhline(
    EXPECTED_FS,
    color=PAIRED_COLOR,
    linestyle="--",
    label=f"Expected ({EXPECTED_FS:g} Hz)",
)
axes[1].set_ylabel("Sampling frequency (Hz)")
axes[1].set_xlabel("Session")
axes[1].set_xticks(positions, acquisition_table["Session"], rotation=90)
axes[1].legend(frameon=False)
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="0.9", linewidth=0.6)
    ax.set_axisbelow(True)
fig.tight_layout()
save_figure(fig, "acquisition_consistency")
plt.show()

## 9. Raw PPG Descriptive Statistics

These acquisition descriptions are not SQI or physiological-quality metrics.

In [ ]:
raw_columns = {
    "file_name": "Session",
    "raw_median": "Median",
    "raw_mean": "Mean",
    "raw_std": "SD",
    "raw_iqr": "IQR",
    "raw_min": "Minimum",
    "raw_max": "Maximum",
    "raw_range": "Range",
}
raw_table = inventory[list(raw_columns)].rename(columns=raw_columns)
raw_dataset_summary = (
    raw_table.drop(columns="Session")
    .agg(["min", "median", "max"])
    .T
    .rename(columns={
        "min": "Session minimum",
        "median": "Session median",
        "max": "Session maximum",
    })
)
display(raw_dataset_summary.style.format("{:.2f}"))
display(raw_table.style.format({
    column: "{:.2f}"
    for column in raw_table.columns
    if column != "Session"
}).hide(axis="index"))

## 10. Core Results for Paper Preparation

Values remain traceable to the inventory and detailed session reports.

In [ ]:
paper_summary = pd.DataFrame({
    "Result": [
        "Audited sessions",
        "Total recording duration (h)",
        "Median session duration (min)",
        "Median estimated sampling frequency (Hz)",
        "Awake / Drowsy duration (h)",
        "Sessions with missing values / time gaps / invalid labels",
        "PASS / WARNING / FAIL",
    ],
    "Value": [
        summary["total_sessions"],
        f"{summary['total_recording_hours']:.2f}",
        f"{summary['session_duration_median'] / 60:.2f}",
        f"{summary['sampling_rate_median']:.2f}",
        (
            f"{summary['total_awake_duration_s'] / 3600:.2f} / "
            f"{summary['total_drowsy_duration_s'] / 3600:.2f}"
        ),
        (
            f"{summary['sessions_with_missing_values']} / "
            f"{summary['sessions_with_time_gaps']} / "
            f"{summary['sessions_with_invalid_labels']}"
        ),
        (
            f"{summary['sessions_pass']} / "
            f"{summary['sessions_warning']} / "
            f"{summary['sessions_fail']}"
        ),
    ],
})
display(paper_summary.style.hide(axis="index"))
display(availability_table.style.hide(axis="index"))

## 11. Reproducible Outputs

The core pipeline writes the inventory, dataset summary, and one JSON report per session. Figures are exported as PDF and 300-dpi PNG. No filtering, SQI, stationarity, peak detection, PPI, or NTSA is performed.

In [ ]:
output_table = pd.DataFrame({
    "Output": [
        "Dataset inventory",
        "Dataset summary",
        "Session reports",
        "Publication figures",
    ],
    "Path": [
        str(audit_result.output_paths["inventory"]),
        str(audit_result.output_paths["summary"]),
        str(audit_result.output_paths["sessions"]),
        str(FIGURE_DIR),
    ],
})
display(output_table.style.hide(axis="index"))